# SceneTwin — Export Description Prediction Tensors

This notebook regenerates the four missing `desc_*_preds.npy` files for the original SceneTwin proof of concept and downloads them in one zip.

Use it when you already have `video_preds.npy` locally and only need the four description tensors for ROI-restricted scoring.


## 1. Runtime

In Colab, switch to a GPU runtime first:

- `Runtime` -> `Change runtime type`
- `Hardware accelerator` -> `GPU`

Run the next install cell once. It intentionally restarts the runtime at the end.


In [ ]:
!rm -rf /content/tribev2
!git clone https://github.com/facebookresearch/tribev2.git
!pip install -e tribev2 --quiet
!pip install huggingface_hub "numpy==2.2.6" --quiet

# Restart runtime to reload numpy cleanly.
import os
os.kill(os.getpid(), 9)


## 2. After Restart

Reconnect to the runtime, then run the remaining cells top to bottom.


In [ ]:
from huggingface_hub import login
login()


In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch

sys.path.insert(0, '/content/tribev2')

from tribev2 import TribeModel

out_dir = Path('/content/scenetwin_desc_preds')
out_dir.mkdir(parents=True, exist_ok=True)

descriptions = {
    'accurate_concise': '''A young woman with short hair stands in a snowy landscape.
She looks determined and sorrowful. Snow falls around her.
The scene is quiet and cold. She begins walking forward through the snow.''',

    'accurate_detailed': '''A young woman with short dark hair stands alone in a vast,
snow-covered landscape under a grey sky. Her expression is a mix of
determination and deep sadness. Large snowflakes drift slowly around her.
She wears layered clothing suitable for harsh winter conditions.
The terrain is flat and featureless, stretching to the horizon.
She takes a breath, then begins walking steadily forward through
the deep snow, leaving footprints behind her.''',

    'bad_vague': '''Something happens on screen. A person is there.
The weather looks cold maybe. They move.''',

    'hallucinated': '''A man in a red suit stands on a tropical beach.
Palm trees sway in warm wind. He is laughing and holding a drink.
Children play in the ocean behind him. The sun is bright and hot.'''
}

for name, text in descriptions.items():
    (out_dir / f'desc_{name}.txt').write_text(text, encoding='utf-8')
    print(f'wrote desc_{name}.txt')

print('torch', torch.__version__)
print('cuda', torch.cuda.is_available())

model = TribeModel.from_pretrained('facebook/tribev2')
print('Model loaded.')

for name in descriptions:
    path = out_dir / f'desc_{name}.txt'
    print(f'\nRunning TRIBE on: {name}')
    events = model.get_events_dataframe(text_path=str(path))
    preds, _ = model.predict(events)
    np.save(out_dir / f'desc_{name}_preds.npy', preds)
    print(f'  saved desc_{name}_preds.npy with shape {preds.shape}')

print('\nSaved files:')
for p in sorted(out_dir.glob('desc_*_preds.npy')):
    print(' ', p.name)


In [ ]:
from google.colab import files

!cd /content/scenetwin_desc_preds && zip -q scenetwin_desc_preds.zip desc_*_preds.npy
files.download('/content/scenetwin_desc_preds/scenetwin_desc_preds.zip')


## 3. What I Need Locally

After download and extraction, the local machine should have these files:

- `video_preds.npy`
- `desc_accurate_concise_preds.npy`
- `desc_accurate_detailed_preds.npy`
- `desc_bad_vague_preds.npy`
- `desc_hallucinated_preds.npy`

With those five tensors, I can run the first ROI-restricted SceneTwin analysis.
